# Step 2-1: Per-Page Render Pipeline

annotated.html → per-page HTML 抽取 → browser render → 單頁 screenshot + bbox map → 對應單頁 source PDF image

每個 source page 各自渲染、各自截圖、各自存 bbox map（座標系統 page-local）。Pair 為純粹的 logical grouping：judge 一次看哪兩頁，靠 `pairs/pair_NN_MM/summary.json` 中的顯式 paths 指向 `pages/page_NN/` 的 artifact。Pair 採 overlap `(1,2), (2,3), ..., (n-1, n)`，page render 去重（每頁只 render 一次）。

In [1]:
# Cell 1: imports
from pathlib import Path
import json
import shutil
import tempfile
from typing import Any

import fitz  # pymupdf
from bs4 import BeautifulSoup, Tag

In [2]:
# Cell 2: config
ROOT = Path("..").resolve()
PDF_PATH = ROOT / "paper" / "JME_DL.pdf"
HTML_PATH = ROOT / "outputs-test-std" / "JME_DL_annotated.html"
PAGE_ORIGIN_MAP_PATH = ROOT / "outputs-test-std" / "JME_DL_page_origin_map.json"
REGION_INDEX_PATH = ROOT / "outputs-test-std" / "JME_DL_region_index.json"
OUT_DIR = ROOT / "outputs-step2"

VIEWPORT = {"width": 1440, "height": 1200}
PDF_ZOOM = 2.0

DRY_RUN_PAGE = 3

In [3]:
page_origin_map = json.loads(PAGE_ORIGIN_MAP_PATH.read_text())
region_index = json.loads(REGION_INDEX_PATH.read_text())
html_text = HTML_PATH.read_text()
soup = BeautifulSoup(html_text, "html.parser")

In [4]:
def make_overlapping_page_pairs(total_pages: int) -> list[tuple[int, int | None]]:
    if total_pages == 0:
        return []
    if total_pages == 1:
        return [(1, None)]
    return [(i, i + 1) for i in range(1, total_pages)]


def make_pair_slug(pair: tuple[int, int | None]) -> str:
    start_page, end_page = pair
    if end_page is None:
        return f"pair_{start_page:02d}"
    return f"pair_{start_page:02d}_{end_page:02d}"


def make_page_slug(source_page: int) -> str:
    return f"page_{source_page:02d}"


def load_annotated_soup(html_path: Path) -> BeautifulSoup:
    html_text = html_path.read_text()
    return BeautifulSoup(html_text, "html.parser")


def extract_page_node(soup: BeautifulSoup, source_page: int) -> Tag:
    target = str(source_page)
    matches = [
        div
        for div in soup.find_all("div", class_="page")
        if div.get("data-source-page") == target
    ]
    if not matches:
        raise ValueError(f"no .page node found for source_page={source_page}")
    if len(matches) > 1:
        raise ValueError(
            f"expected exactly 1 .page node for source_page={source_page}, "
            f"got {len(matches)}"
        )
    return matches[0]


def build_page_html(
    original_soup: BeautifulSoup,
    page_node: Tag,
    title: str,
) -> str:
    original_head = original_soup.find("head")
    if original_head is None:
        raise ValueError("annotated HTML is missing <head>")

    pair_soup = BeautifulSoup(
        "<!DOCTYPE html>\n<html><head></head><body></body></html>",
        "html.parser",
    )

    original_html = original_soup.find("html")
    if original_html is not None:
        pair_soup.html.attrs.update(original_html.attrs)

    for node in original_head.contents:
        fragment = BeautifulSoup(str(node), "html.parser")
        for child in list(fragment.contents):
            pair_soup.head.append(child)

    if pair_soup.title is None:
        title_tag = pair_soup.new_tag("title")
        title_tag.string = title
        pair_soup.head.append(title_tag)
    else:
        pair_soup.title.string = title

    fragment = BeautifulSoup(str(page_node), "html.parser")
    page = fragment.find("div", class_="page")
    if page is None:
        raise ValueError("page_node must be a .page element")
    pair_soup.body.append(page)
    pair_soup.body.append(pair_soup.new_string("\n"))

    return str(pair_soup)


def save_page_html(
    html: str,
    out_dir: Path,
    source_page: int,
) -> Path:
    page_slug = make_page_slug(source_page)
    page_dir = out_dir / "pages" / page_slug
    page_dir.mkdir(parents=True, exist_ok=True)
    page_html_path = page_dir / "rendered.html"
    page_html_path.write_text(html, encoding="utf-8")
    return page_html_path


def render_pdf_page(
    pdf_path: Path,
    source_page: int,
    out_dir: Path,
    zoom: float = 2.0,
    pdf_doc: "fitz.Document | None" = None,
) -> Path:
    page_slug = make_page_slug(source_page)
    page_dir = out_dir / "pages" / page_slug
    page_dir.mkdir(parents=True, exist_ok=True)
    output_path = page_dir / "source_pdf.png"

    matrix = fitz.Matrix(zoom, zoom)

    def _render(doc: "fitz.Document") -> None:
        # PyMuPDF is 0-based; schema is 1-based
        page = doc.load_page(source_page - 1)
        pix = page.get_pixmap(matrix=matrix)
        pix.save(str(output_path))

    if pdf_doc is not None:
        _render(pdf_doc)
    else:
        with fitz.open(pdf_path) as owned_doc:
            _render(owned_doc)
    return output_path

In [5]:
from pydantic import BaseModel, Field


class Viewport(BaseModel):
    width: int
    height: int


class PagePixelSize(BaseModel):
    width: int
    height: int


class DOMBBoxElement(BaseModel):
    anchor_id: str = Field(description="對應 HTML 裡的 data-region-id")
    source_page: int = Field(description="原始 PDF 頁碼，作為 defensive check 對應 header")
    tag: str = Field(description="DOM tag，例如 div, p, h2")
    text_preview: str = Field(description="元素文字摘要，方便 debug")
    bbox: list[float] = Field(
        min_length=4,
        max_length=4,
        description="[x, y, w, h]，以該頁 screenshot 的 pixel 座標表示",
    )


class DOMBBoxHeader(BaseModel):
    source_page: int = Field(description="此 bbox map 對應的單一原始 PDF 頁碼")
    page_slug: str = Field(description="例如 page_03")
    html_path: str = Field(description="該頁 rendered.html 的相對路徑")
    screenshot_path: str = Field(description="該頁 rendered_screenshot.png 的相對路徑")
    viewport: Viewport = Field(description="render 時 Playwright 用的 viewport")
    page_pixel_size: PagePixelSize = Field(description="full-page screenshot 實際像素尺寸")
    bbox_count: int


class DOMBBoxMap(BaseModel):
    dom_header: DOMBBoxHeader
    dom: list[DOMBBoxElement]

In [6]:
from playwright.async_api import async_playwright, Browser


async def capture_page_screenshot_and_bboxes(
    html_file: str | Path,
    out_dir: Path,
    source_page: int,
    viewport: dict[str, int],
    browser: Browser | None = None,
) -> dict[str, Any]:
    html_file = Path(html_file).resolve()
    out_dir = Path(out_dir).resolve()
    page_slug = make_page_slug(source_page)
    page_dir = out_dir / "pages" / page_slug
    page_dir.mkdir(parents=True, exist_ok=True)

    screenshot_path = page_dir / "rendered_screenshot.png"
    bbox_map_path = page_dir / "dom_bbox_map.json"

    def rel(path: Path) -> str:
        try:
            return str(path.relative_to(ROOT))
        except ValueError:
            return str(path)

    async def _do_capture(browser_: Browser) -> dict[str, Any]:
        page = await browser_.new_page(viewport=viewport)
        try:
            await page.goto(html_file.as_uri(), wait_until="load")
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(200)

            full_page_size = await page.evaluate("""
            () => {
                const body = document.body;
                const doc = document.documentElement;
                return {
                    width: Math.max(body.scrollWidth, doc.scrollWidth, body.clientWidth, doc.clientWidth),
                    height: Math.max(body.scrollHeight, doc.scrollHeight, body.clientHeight, doc.clientHeight),
                };
            }
            """)

            raw_dom = await page.locator("[data-region-id]").evaluate_all("""
            els => els.map(el => {
                const rect = el.getBoundingClientRect();
                return {
                    anchor_id: el.getAttribute("data-region-id"),
                    source_page: Number(el.getAttribute("data-source-page")),
                    tag: el.tagName.toLowerCase(),
                    text_preview: (el.innerText || el.textContent || "").trim().replace(/\\s+/g, " ").slice(0, 100),
                    bbox: [rect.x, rect.y, rect.width, rect.height],
                };
            })
            """)

            dom = [DOMBBoxElement.model_validate(item).model_dump() for item in raw_dom]

            await page.screenshot(path=str(screenshot_path), full_page=True)

            screenshot_width = int(full_page_size["width"])
            screenshot_height = int(full_page_size["height"])

            dom_bbox_map = DOMBBoxMap(
                dom_header=DOMBBoxHeader(
                    source_page=source_page,
                    page_slug=page_slug,
                    html_path=rel(html_file),
                    screenshot_path=rel(screenshot_path),
                    viewport=Viewport(**viewport),
                    page_pixel_size=PagePixelSize(
                        width=screenshot_width,
                        height=screenshot_height,
                    ),
                    bbox_count=len(dom),
                ),
                dom=dom,
            )
            bbox_map_path.write_text(
                json.dumps(dom_bbox_map.model_dump(), ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            return {
                "source_page": source_page,
                "page_slug": page_slug,
                "html_path": rel(html_file),
                "screenshot_path": rel(screenshot_path),
                "viewport": dict(viewport),
                "page_pixel_size": {
                    "width": screenshot_width,
                    "height": screenshot_height,
                },
                "bbox_count": len(dom),
                "bbox_map_path": rel(bbox_map_path),
            }
        finally:
            await page.close()

    if browser is not None:
        return await _do_capture(browser)

    async with async_playwright() as p:
        owned_browser = await p.chromium.launch(headless=True)
        try:
            return await _do_capture(owned_browser)
        finally:
            await owned_browser.close()

In [7]:
async def run_page_pipeline(
    soup: BeautifulSoup,
    pdf_path: Path,
    source_page: int,
    out_dir: Path,
    viewport: dict[str, int],
    pdf_zoom: float,
    browser: Browser | None = None,
    pdf_doc: "fitz.Document | None" = None,
) -> dict[str, Any]:
    page_slug = make_page_slug(source_page)
    page_node = extract_page_node(soup, source_page)
    page_html = build_page_html(soup, page_node, title=page_slug)
    page_html_path = save_page_html(page_html, out_dir, source_page)
    pdf_image_path = render_pdf_page(
        pdf_path, source_page, out_dir, zoom=pdf_zoom, pdf_doc=pdf_doc
    )

    capture = await capture_page_screenshot_and_bboxes(
        html_file=page_html_path,
        out_dir=out_dir,
        source_page=source_page,
        viewport=viewport,
        browser=browser,
    )

    def rel(path: Path) -> str:
        try:
            return str(path.resolve().relative_to(ROOT))
        except ValueError:
            return str(path)

    return {
        "source_page": source_page,
        "page_slug": page_slug,
        "rendered_html_path": rel(page_html_path),
        "rendered_screenshot_path": capture["screenshot_path"],
        "source_pdf_path": rel(pdf_image_path),
        "dom_bbox_map_path": capture["bbox_map_path"],
        "region_count": capture["bbox_count"],
    }


def build_and_save_pair_summary(
    pair: tuple[int, int | None],
    page_artifacts: dict[int, dict[str, Any]],
    out_dir: Path,
) -> Path:
    pair_slug = make_pair_slug(pair)
    pair_dir = out_dir / "pairs" / pair_slug
    pair_dir.mkdir(parents=True, exist_ok=True)

    pages_payload = []
    for sp in pair:
        if sp is None:
            continue
        artifact = page_artifacts[sp]
        pages_payload.append({
            "source_page": artifact["source_page"],
            "page_slug": artifact["page_slug"],
            "rendered_html_path": artifact["rendered_html_path"],
            "rendered_screenshot_path": artifact["rendered_screenshot_path"],
            "source_pdf_path": artifact["source_pdf_path"],
            "dom_bbox_map_path": artifact["dom_bbox_map_path"],
            "region_count": artifact["region_count"],
        })

    summary = {
        "pair": list(pair),
        "pair_slug": pair_slug,
        "pages": pages_payload,
    }
    summary_path = pair_dir / "summary.json"
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return summary_path


def save_all_pairs_summary(
    unique_pages: list[int],
    pairs: list[tuple[int, int | None]],
    page_artifacts: dict[int, dict[str, Any]],
    source_pdf_path: Path,
    annotated_html_path: Path,
    out_dir: Path,
) -> Path:
    def rel(path: Path) -> str:
        try:
            return str(path.resolve().relative_to(ROOT))
        except ValueError:
            return str(path)

    payload = {
        "total_source_pages": len(unique_pages),
        "source_pdf_path": rel(source_pdf_path),
        "annotated_html_path": rel(annotated_html_path),
        "pages": [
            {
                "source_page": sp,
                "page_slug": page_artifacts[sp]["page_slug"],
                "dom_bbox_map_path": page_artifacts[sp]["dom_bbox_map_path"],
            }
            for sp in unique_pages
        ],
        "pairs": [
            {
                "pair": list(pair),
                "pair_slug": make_pair_slug(pair),
                "summary_path": f"outputs-step2/pairs/{make_pair_slug(pair)}/summary.json",
            }
            for pair in pairs
        ],
    }
    out_path = out_dir / "all_pairs_summary.json"
    out_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return out_path

In [8]:
# Single-page dry run (page DRY_RUN_PAGE)
sp = load_annotated_soup(HTML_PATH)
dry_run_result = await run_page_pipeline(
    soup=sp,
    pdf_path=PDF_PATH,
    source_page=DRY_RUN_PAGE,
    out_dir=OUT_DIR,
    viewport=VIEWPORT,
    pdf_zoom=PDF_ZOOM,
)
dry_run_result

MuPDF error: format error: No common ancestor in structure tree



{'source_page': 3,
 'page_slug': 'page_03',
 'rendered_html_path': 'outputs-step2/pages/page_03/rendered.html',
 'rendered_screenshot_path': 'outputs-step2/pages/page_03/rendered_screenshot.png',
 'source_pdf_path': 'outputs-step2/pages/page_03/source_pdf.png',
 'dom_bbox_map_path': 'outputs-step2/pages/page_03/dom_bbox_map.json',
 'region_count': 12}

In [9]:
# Batch orchestration: cleanup -> render all unique pages -> build pair summaries -> top-level index
# WARNING: clears entire OUT_DIR (outputs-step2/). Don't put manual artifacts here.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

sp = load_annotated_soup(HTML_PATH)

# Stage 1: list unique pages, render each once (shared browser + shared fitz doc)
page_artifacts: dict[int, dict[str, Any]] = {}
with fitz.open(PDF_PATH) as pdf_doc:
    total_pages = pdf_doc.page_count
    pairs = make_overlapping_page_pairs(total_pages)
    unique_pages = sorted({p for pair in pairs for p in pair if p is not None})

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        try:
            for source_page in unique_pages:
                page_artifacts[source_page] = await run_page_pipeline(
                    soup=sp,
                    pdf_path=PDF_PATH,
                    source_page=source_page,
                    out_dir=OUT_DIR,
                    viewport=VIEWPORT,
                    pdf_zoom=PDF_ZOOM,
                    browser=browser,
                    pdf_doc=pdf_doc,
                )
        finally:
            await browser.close()

# Stage 2: build pair summaries (pure reference assembly)
for pair in pairs:
    build_and_save_pair_summary(pair, page_artifacts, OUT_DIR)

# Stage 3: top-level index
all_pairs_summary_path = save_all_pairs_summary(
    unique_pages=unique_pages,
    pairs=pairs,
    page_artifacts=page_artifacts,
    source_pdf_path=PDF_PATH,
    annotated_html_path=HTML_PATH,
    out_dir=OUT_DIR,
)
{
    "total_pages": total_pages,
    "unique_pages_rendered": len(page_artifacts),
    "pairs_built": len(pairs),
    "all_pairs_summary_path": str(all_pairs_summary_path.relative_to(ROOT)),
}

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error

{'total_pages': 26,
 'unique_pages_rendered': 26,
 'pairs_built': 25,
 'all_pairs_summary_path': 'outputs-step2/all_pairs_summary.json'}

In [10]:
# Visual check: side-by-side PDF page vs HTML render for DRY_RUN_PAGE
from IPython.display import Image, display, HTML as IPyHTML

page_slug = make_page_slug(DRY_RUN_PAGE)
page_dir = OUT_DIR / "pages" / page_slug
pdf_img = page_dir / "source_pdf.png"
html_img = page_dir / "rendered_screenshot.png"

print(f"=== {page_slug} ===")
print(f"PDF:  {pdf_img.relative_to(ROOT)}")
print(f"HTML: {html_img.relative_to(ROOT)}")
display(IPyHTML(f"""
<div style="display: flex; gap: 12px;">
  <div style="flex: 1;"><b>source_pdf.png</b><br><img src="{pdf_img.as_uri()}" style="width: 100%;"></div>
  <div style="flex: 1;"><b>rendered_screenshot.png</b><br><img src="{html_img.as_uri()}" style="width: 100%;"></div>
</div>
"""))

bbox_map = json.loads((page_dir / "dom_bbox_map.json").read_text())
print(f"bbox_count: {bbox_map['dom_header']['bbox_count']}")
print(f"page_pixel_size: {bbox_map['dom_header']['page_pixel_size']}")

=== page_03 ===
PDF:  outputs-step2/pages/page_03/source_pdf.png
HTML: outputs-step2/pages/page_03/rendered_screenshot.png


bbox_count: 12
page_pixel_size: {'width': 1440, 'height': 1696}
